# Seed & Initialization

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load Data

In [2]:
from datasets import load_dataset 

chat_doctor = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

print(chat_doctor.column_names)

['instruction', 'input', 'output']


In [3]:
print(chat_doctor)
print(chat_doctor[0])
print(chat_doctor[1])

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 112165
})
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common sympto

## Format the data

In [4]:
tokenizer = AutoTokenizer.from_pretrained("qwen-medical-pretrained") # load tokenizer from cpt


def format_chatdoctor(example):
    messages = [
        {"role": "system", "content": example["instruction"]}, 
        {"role": "user", "content": example["input"]}, 
        {"role": "assistant", "content": example["output"]},
    ]

    # auto convert to Qwen format or any model 
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False,
    )

    return {"text": text}

chat_doctor = chat_doctor.map(
    format_chatdoctor, 
    remove_columns=chat_doctor.column_names,
)

In [5]:
print(chat_doctor)
# print(chat_doctor[0])
print(chat_doctor[0]["text"])

Dataset({
    features: ['text'],
    num_rows: 112165
})
<|im_start|>system
If you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>
<|im_start|>user
I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!<|im_end|>
<|im_start|>assistant
Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most com

In [6]:
# Split dataset into train / eval (5%)
SEED = 42

# Shuffle then split to ensure random distribution
sft_data = chat_doctor.shuffle(seed=SEED)
split = sft_data.train_test_split(test_size=0.05, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train size: {len(train_dataset)}, Eval size: {len(eval_dataset)}")
# show one example for quick sanity check
print("--- sample train text (truncated) ---")
print(train_dataset[0]["text"][:400])

Train size: 106556, Eval size: 5609
--- sample train text (truncated) ---
<|im_start|>system
If you are a doctor, please answer the medical questions based on the patient's description.<|im_end|>
<|im_start|>user
My father was treated for prostate cancer and was given radioactive seeds. The process was quite a few years ago when the seeds were new. He was grossly overdosed and is suffering with the damages of the radiation. We would like to find a doctor anywhere in the


# USING TRL LIBRARY

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch 
import wandb

# Login to W&B (if not already logged in)
# wandb.login()

# Load tokenizer and model (fall back to base if local checkpoint invalid)
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
try:
    model = AutoModelForCausalLM.from_pretrained(
        "qwen-medical-pretrained",
        torch_dtype=torch.bfloat16,
    )
    print("✓ Loaded CPT checkpoint: qwen-medical-pretrained")
except Exception as e:
    print("⚠ Failed to load local checkpoint, loading base model instead:", e)
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.bfloat16)

# SFT config with W&B logging - optimized for RTX 5070 Ti (15GB VRAM)
sft_args = SFTConfig(
    output_dir="qwen-medical-sft",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,  # Effective batch = 4

    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_steps=60,

    num_train_epochs=3,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    logging_steps=10,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",  # Reduce optimizer memory
    torch_empty_cache_steps=5,   # Clear cache more frequently
    
    # Skip eval during training to save memory
    # evaluation_strategy="no",
    
    # W&B logging
    report_to="wandb",
    run_name="sft-qwen-medical-chatdoctor",
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✓ Loaded CPT checkpoint: qwen-medical-pretrained


In [8]:
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [9]:
# Clear GPU cache and reinitialize trainer with smaller batch size
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
print("✓ GPU cache cleared")

# Reinitialize trainer with new config
trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)
print("✓ Trainer reinitialized with optimized batch size (1 per GPU, accumulation 4)")

✓ GPU cache cleared
✓ Trainer reinitialized with optimized batch size (1 per GPU, accumulation 4)


In [10]:
# Start training (will log to W&B and save checkpoints)
print("Starting SFT training...")
trainer.train()
print("✓ Training completed!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting SFT training...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/pc5070ti/.netrc.
wandb: Currently logged in as: nguyentranhoangnhan18 (nguyentranhoangnhan18-ton-duc-thang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
10,3.348568
20,3.179555
30,2.955701
40,2.921950
50,2.873540
60,2.802028
70,2.727434
80,2.605027
90,2.736484


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x70b260795670>> (for post_run_cell), with arguments args (<ExecutionResult object at 70b02388a630, execution_count=10 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 70b02388a5d0, raw_cell="# Start training (will log to W&B and save checkpo.." transformed_cell="# Start training (will log to W&B and save checkpo.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B100.110.165.40/home/pc5070ti/workspace/LLM-Learning/SFT.ipynb#X41sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

# Tokenize

# Create Label 
(SFT need this, different from CPT)


# Split Train/Test

# Data Collator
(Different from CPT)

# Load Model CPT

# Training Arguments

# Trainer & Training

# Save Model

# Test Generation